In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_txn = pd.read_csv('../data/raw/train_transaction.csv')
train_id  = pd.read_csv('../data/raw/train_identity.csv')

df = train_txn.merge(train_id, on='TransactionID', how='left')

In [3]:
y = df['isFraud'].copy()
df = df.drop(columns=['isFraud', 'TransactionID'])

In [4]:
threshold = 0.90
missing_pct = df.isnull().mean()

cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
df = df.drop(columns=cols_to_drop)

print(f"Columns dropped (>90% missing): {len(cols_to_drop)}")
print(f"Remaining columns: {df.shape[1]}")

Columns dropped (>90% missing): 12
Remaining columns: 420


In [5]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=np.number).columns.tolist()

In [6]:
print(f"Categorical columns: {len(cat_cols)} → {cat_cols}")
print(f"Numerical columns:   {len(num_cols)}")

Categorical columns: 29 → ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
Numerical columns:   391


In [7]:
# Numerical → median
for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical → 'Unknown'
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

print(f"Missing values remaining: {df.isnull().sum().sum()}")

Missing values remaining: 0


In [9]:
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(df[cat_cols].head(3))      

   ProductCD  card4  card6  P_emaildomain  R_emaildomain  M1  M2  M3  M4  M5  \
0          4      2      2              0              0   1   1   1   2   0   
1          4      3      2              9              0   2   2   2   0   1   
2          4      4      3             30              0   1   1   1   0   0   

   ...  id_30  id_31  id_33  id_34  id_35  id_36  id_37  id_38  DeviceType  \
0  ...     36     37    180      0      2      2      2      2           0   
1  ...     36     37    180      0      2      2      2      2           0   
2  ...     36     37    180      0      2      2      2      2           0   

   DeviceInfo  
0         621  
1         621  
2         621  

[3 rows x 29 columns]


In [10]:
# 1. Time-based features
#    TransactionDT is seconds offset from a reference time
df['hour']    = (df['TransactionDT'] // 3600) % 24
df['day']     = (df['TransactionDT'] // (3600 * 24)) % 7
df['week']    = (df['TransactionDT'] // (3600 * 24 * 7)) % 52

# 2. Log transform on transaction amount
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

# 3. Aggregation features — mean & std amount per card
for col in ['card1', 'card2', 'card3']:
    df[f'{col}_amt_mean'] = df.groupby(col)['TransactionAmt'].transform('mean')
    df[f'{col}_amt_std']  = df.groupby(col)['TransactionAmt'].transform('std').fillna(0)

# 4. Frequency encoding — how often does each card/email appear?
for col in ['card1', 'card2', 'P_emaildomain']:
    freq = df[col].value_counts(normalize=True)
    df[f'{col}_freq'] = df[col].map(freq)

# 5. Amount deviation from card mean (useful fraud signal)
df['amt_deviation'] = df['TransactionAmt'] - df['card1_amt_mean']

print(f"New features added. Final shape: {df.shape}")
print("New columns:", ['hour','day','week','TransactionAmt_log',
      'card1_amt_mean','card1_amt_std','card1_freq','amt_deviation'])

New features added. Final shape: (590540, 434)
New columns: ['hour', 'day', 'week', 'TransactionAmt_log', 'card1_amt_mean', 'card1_amt_std', 'card1_freq', 'amt_deviation']


In [11]:
import os
os.makedirs('../data/processed', exist_ok=True)

df['isFraud'] = y.values

df.to_csv('../data/processed/train_processed.csv', index=False)

print(f"Final shape: {df.shape}")
print(f"Fraud rate preserved: {df['isFraud'].mean()*100:.2f}%")

Final shape: (590540, 435)
Fraud rate preserved: 3.50%
